Axioms for the theory of <a class="ProveItLink" href="theory.ipynb">proveit.physics.quantum.QEC2</a>
========

In [ ]:
import proveit
# Prepare this notebook for defining the axioms of a theory:
%axioms_notebook # Keep this at the top following 'import proveit'.

from proveit                 import (e, f, i, l, n, s, A, B, X,
                                     ExprRange, ExprTuple, IndexedVar)
from proveit.core_expr_types import A_1_to_n
from proveit.logic           import Equals, Exists, Forall, InSet
from proveit.logic.sets      import EmptySet, Intersect, Union, Disjoint
from proveit.numbers         import (zero, one, three, Add, greater_eq,
                                     LessEq, Mult, Natural, Neg)
from proveit.linear_algebra  import AntiCommutator

from proveit.physics.quantum.QEC2 import (
        _ell, _max_buf_weight, ActionFunction, BufiloSequences, CheckFunction,
        Errors, f_one_to_n, Faults, State, States, Weight)


In [ ]:
%begin axioms

In [ ]:
weight_in_natural = Forall(A, InSet(Weight(A), Natural))

In [ ]:
weight_empty_set = Equals(Weight(EmptySet), zero)

In [ ]:
Add(ExprRange(i, Weight(IndexedVar(A, i)), one, n))

In [ ]:
weight_additivity = Forall(n,
       Forall(A_1_to_n,
              Equals(Weight(Union(A_1_to_n)),
                     Add(ExprRange(i, Weight(IndexedVar(A, i)), one, n))),
       conditions=[Disjoint(A_1_to_n)]),
domain=Natural)

In [ ]:
binary_weight_additivity = (
    Forall((A, B),
           Equals(Weight(Union(A, B)),
                          Add(Weight(A), Weight(B), Neg(Weight(Intersect(A, B)))))
    )
)

##### Axiomatic Characterization of BUFILO weight limit $w_{\text{BUF}}$

In [ ]:
_max_buf_weight_in_natural = InSet(_max_buf_weight, Natural)

In [ ]:
_max_buf_weight_ge_three = greater_eq(_max_buf_weight, three)

##### Axiomatic Characterization of BUFILO Sequence, $f \in \mathcal{F}_{\ell, w_{\text{BUF}}}^{\text{seq}}$

In [ ]:
anti_commutation_buf_seq_first_elem = (
    Forall(n,
    Forall(f_one_to_n,
           Equals(AntiCommutator(IndexedVar(f, one), _ell),
                  zero),
    condition=InSet(ExprTuple(f_one_to_n), BufiloSequences), domain=Faults),
    conditions = [LessEq(n, _max_buf_weight)], domain = Natural)
)

#### Augmented Syndrome States

In [ ]:
state_membership_def = Forall(s, Equals(InSet(s, States), Exists(e, Equals(s, State(_ell, e)), domain = Errors)))

In [ ]:
state_def = (
    Forall(e, Equals(State(_ell, e), ExprTuple(CheckFunction(e), ActionFunction(_ell,e))),
           domain=Errors
          )
)

#### The All-Syndromes Graph

$\text{all\_syndromes\_graph} = \text{Graph}(V, E)$, where:

$V = \text{States}$ (_i.e._, the set of all “augmented syndrome states” $S_{\ell}$

$E = \{(n_{1}, n_{2}) \,|\, \big[A_{\ell}(n_{1})=0 \land A_{\ell}(n_{2})=1\big] \lor \big[A_{\ell}(n_{1})=1 \land \nu(\text{syn}(n_{1})) \in \text{syn}(n_{1}) - \text{syn}(n_{2})\big]\}$

and $A_{\ell}$ is the “action matrix” or “action function” with respect to logical operator $\ell$, and $\text{syn}(n)$ represents the syndrome component of the augmented syndrome state $n$.

Questions:

1. Is $(n_{1}, n_{2})$ a directed edge?

Looks like we need something else here. WW tried something like $A(n_{1}) = 0$, but the $n_{1}$ is a node in the graph, and the graph nodes are _states_, not errors.

In [ ]:
from proveit import s, X, Y, Function
from proveit.logic import And, Or
from proveit.logic.sets import Difference, Set, SetOfAll
from proveit.graphs import Graph
from proveit.physics.quantum.QEC2 import (
        _nu, ActionFunction, s_prime, StateAction, States, StateSyndrome)
all_states_graph = Graph(States, SetOfAll((s, s_prime), Set(s, s_prime),
         conditions=[Or(
             And(Equals(StateAction(s), zero), Equals(StateAction(s_prime), one)),
             And(Equals(StateAction(s), one),
                 InSet(Function(_nu, StateSyndrome(s)), Difference(StateSyndrome(s), StateSyndrome(s_prime))))
         ).with_wrap_after_operator()],
         domain=States))

In [ ]:
%end axioms